# Document Analysis & Knowledge-Base Assistant — RAG Mini-Study
**SafeX Solutions — AI Agent Automation Proposal (Week 3)**
**Developer:** Ali Zaib | AI/ML Intern

**Target company:** QuickBite — an original, fictional food-delivery company
standing in for Foodpanda (per the assignment's substitution note). All 5
knowledge-base documents below are original content written for this
exercise, not sourced from any real company's actual policies.

This notebook walks through the full RAG pipeline end-to-end:

```
Documents -> Chunking -> TF-IDF Embedding -> Vector Store
          -> Query -> Top-K Retrieval -> Similarity threshold check
          -> Abstain, OR Answer Synthesis -> Response
```


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))  # repo root
import pandas as pd

from src.modules.doc_knowledge_assistant.engine import DocKnowledgeAssistantEngine, SIMILARITY_THRESHOLD
from src.modules.doc_knowledge_assistant.test_suite import TEST_SUITE

engine = DocKnowledgeAssistantEngine()
print("Documents indexed:", engine.document_titles())
print("Total chunks:", len(engine.chunks))
print("Abstention similarity threshold:", SIMILARITY_THRESHOLD)


Documents indexed: ['QuickBite Delivery Policy', 'QuickBite Payment Methods FAQ', 'QuickBite Refund and Cancellation Policy', 'QuickBite Restaurant Partner FAQ', 'QuickBite Rider Partner FAQ']
Total chunks: 35
Abstention similarity threshold: 0.12


## 1. The knowledge base
Each document is chunked by its `## ` section headings.

In [2]:
chunk_counts = pd.Series([c.doc_title for c in engine.chunks]).value_counts()
chunk_counts


QuickBite Delivery Policy                   7
QuickBite Payment Methods FAQ               7
QuickBite Refund and Cancellation Policy    7
QuickBite Restaurant Partner FAQ            7
QuickBite Rider Partner FAQ                 7
Name: count, dtype: int64

In [3]:
# Sample chunk
sample = engine.chunks[0]
print("Document:", sample.doc_title)
print("Section:", sample.section)
print("Text:", sample.text[:200], "...")


Document: QuickBite Delivery Policy
Section: Delivery Areas and Coverage
Text: QuickBite currently delivers within a 12-kilometer radius of each partner restaurant in supported cities (Islamabad, Lahore, Karachi, and Rawalpindi). You can check whether your address is covered by  ...


## 2. Retrieval
Given a query, the engine TF-IDF-embeds it and ranks all chunks by cosine
similarity. Section headings are boosted into each chunk's embedding text
(repeated twice) before vectorizing — this was found during development to
meaningfully improve retrieval accuracy (see Section 5).

In [4]:
query = "How much do riders get paid per delivery?"
results = engine.retrieve(query, top_k=3)
for r in results:
    print(f"{r.score:.3f}  {r.chunk.doc_title} :: {r.chunk.section}")


0.286  QuickBite Rider Partner FAQ :: How Riders Are Paid
0.083  QuickBite Payment Methods FAQ :: Loyalty Points
0.077  QuickBite Delivery Policy :: Contactless and In-Person Delivery


## 3. Answer synthesis
The top-matching chunk's own text is returned, with its source cited.

In [5]:
result = engine.answer(query)
print(result.answer)
print()
print("Abstained:", result.abstained)


Riders earn a base fee per delivery (distance-based, PKR 60-140) plus 100% of any customer tip. Peak-hour deliveries (12-2 PM, 7-9 PM) earn an additional PKR 20-40 surge bonus per order. Weekly earnings are deposited directly to the rider's bank account or JazzCash/EasyPaisa account every Monday for the previous Monday-Sunday period.

_Source: QuickBite Rider Partner FAQ — "How Riders Are Paid"_

Abstained: False


## 4. Handling out-of-scope questions (hallucination mitigation)
If the best retrieval score falls below `SIMILARITY_THRESHOLD`, the engine
abstains instead of guessing. This is the module's main hallucination
control: because answers are always extracted from a retrieved chunk (never
generated freely by default), the assistant literally cannot say something
that isn't grounded in the knowledge base — except in the case where it
matches the *wrong* document confidently enough to clear the threshold,
which is the real remaining risk (see Section 6).

In [6]:
out_of_scope = engine.answer("What is the weather like in Islamabad today?")
print(out_of_scope.answer)
print()
print("Abstained:", out_of_scope.abstained)
print("Top retrieved score:", out_of_scope.retrieved[0].score, "(below threshold)")


I don't have information about that in the QuickBite knowledge base I was given. Please contact support directly for this question, or rephrase it if you think it should be covered.

Abstained: True
Top retrieved score: 0.087 (below threshold)


## 5. Full evaluation: 14 sample questions (12 answerable + 2 out-of-scope)

In [7]:
report = engine.run_evaluation(TEST_SUITE)
print(f"Accuracy: {report['accuracy_percent']}% ({report['correct']}/{report['total']})")

results_df = pd.DataFrame(report["results"])
results_df["expected_doc"] = results_df["expected_doc"].fillna("(out of scope)")
results_df


Accuracy: 100.0% (14/14)


,query,expected_doc,abstained,top_doc,top_score,passed
0,How long does standard delivery usually take?,QuickBite Delivery Policy,False,QuickBite Delivery Policy,0.273,True
1,What happens if my order is late?,QuickBite Delivery Policy,False,QuickBite Delivery Policy,0.217,True
2,Can I cancel my order for free right after pla...,QuickBite Refund and Cancellation Policy,False,QuickBite Refund and Cancellation Policy,0.400,True
3,How do I get a refund if an item is missing fr...,QuickBite Refund and Cancellation Policy,False,QuickBite Refund and Cancellation Policy,0.276,True
4,What payment methods do you accept?,QuickBite Payment Methods FAQ,False,QuickBite Payment Methods FAQ,0.295,True
5,Is there a limit on cash on delivery orders?,QuickBite Payment Methods FAQ,False,QuickBite Payment Methods FAQ,0.329,True
6,How do loyalty points work?,QuickBite Payment Methods FAQ,False,QuickBite Payment Methods FAQ,0.477,True
7,What do I need to become a delivery rider?,QuickBite Rider Partner FAQ,False,QuickBite Rider Partner FAQ,0.182,True
8,How much do riders get paid per delivery?,QuickBite Rider Partner FAQ,False,QuickBite Rider Partner FAQ,0.286,True
9,How can my restaurant join QuickBite?,QuickBite Restaurant Partner FAQ,False,QuickBite Restaurant Partner FAQ,0.204,True


### Why heading-boosting mattered

During development, the *first* version of the embedding step used only
each chunk's raw body text (no heading boost). That version scored **71.4%**
on this same test suite — several answerable questions were retrieved from
the *wrong* document. Investigating why revealed a classic TF-IDF weakness:
the word "delivery" appears in nearly every chunk across all 5 documents
(since the whole knowledge base is about a delivery company), so its IDF
weight is low and it barely helps distinguish anything. Paraphrased
questions like *"how long does delivery **usually take**"* shared almost no
other vocabulary with the correct chunk's actual wording (*"delivered
**within 30-45 minutes**"*), so the match failed.

Repeating each chunk's section heading into its embedding text (e.g. "Standard
Delivery Times. Standard Delivery Times. Most orders are delivered
within...") gives short, topical heading words much more weight relative to
generic body vocabulary, which raised accuracy to 100% on this test suite.
This is a real, standard RAG technique (heading/title-boosted embeddings),
not a test-specific hack — though see the limitations below for why 100%
on 14 questions doesn't mean the retriever is perfect.

## 6. Known limitations

- **Lexical, not semantic, matching.** TF-IDF only recognizes shared words
  and phrases. A true embedding model (e.g. sentence-transformers) would
  generalize across paraphrases much better than the heading-boost
  workaround above.
- **Abstention is a similarity-score heuristic, not true uncertainty
  estimation.** A borderline or ambiguous question could still score above
  the threshold against the *wrong* document, in which case the assistant
  would confidently return an irrelevant (though still grounded-in-a-real-
  chunk) answer, rather than abstaining. This is the main residual
  hallucination-adjacent risk in this design.
- **Extractive answers only, by default.** Since there's no generative LLM
  in the default `mock` mode, answers can't combine information across
  multiple chunks the way a generative model could — they're always a
  direct quote of the best-matching chunk (plus one related chunk, if it
  also clears the threshold).
- **Small knowledge base (5 documents, 35 chunks).** Retrieval accuracy at
  this scale doesn't guarantee the same performance on a much larger,
  more topically overlapping document set, where more chunks would compete
  for similar vocabulary.
- **Optional LLM generation path is untested against a live API** in this
  evaluation environment (no API key configured) — `engine.py` implements
  `openai` and `gemini` providers behind the same interface, but only the
  default `mock` (extractive) path has been run and evaluated here.
